In [1]:
import os
import pandas as pd
# from google.colab import drive

In [2]:
# drive.mount('/content/drive')
# os.chdir('/content/drive/MyDrive/Real-Time UAV Fault Detection and Classification Using Time-Series Deep Learning on Embedded Edge Devices/')

In [ ]:
df = pd.read_parquet("../processed/ALFA_Dataset_Full.parquet")
df = df.sort_values(by='recording_timestamp')

In [4]:
pd.set_option('display.max_columns', None)
df.info()

<class 'pandas.DataFrame'>
Index: 692004 entries, 598160 to 114088
Columns: 447 entries, recording_timestamp to elevator_failure_status
dtypes: float64(331), int64(2), str(114)
memory usage: 3.1 GB


In [5]:
# Đếm tổng số cột có chứa ít nhất một giá trị NaN/Null
num_missing_cols = df.isnull().any().sum()

print(f"Tổng số cột bị khuyết thiếu dữ liệu: {num_missing_cols} cột trên tổng số {df.shape[1]} cột.")

Tổng số cột bị khuyết thiếu dữ liệu: 442 cột trên tổng số 447 cột.


In [6]:
nan_percentage = (df.isnull().sum() / len(df)) * 100

# Chỉ hiển thị các cột có chứa giá trị thiếu (sắp xếp giảm dần)
missing_info = nan_percentage[nan_percentage > 0].sort_values(ascending=False)
print("Các cột bị thiếu dữ liệu và tỷ lệ % khuyết thiếu:")
print(missing_info.head(30)) # Xem nhanh 30 cột thiếu nhiều nhất

Các cột bị thiếu dữ liệu và tỷ lệ % khuyết thiếu:
battery_design_capacity         100.000000
frame_id_out                    100.000000
frame_id_time_reference         100.000000
frame_id_state                  100.000000
frame_id_diagnostics            100.000000
frame_id_battery                100.000000
frame_id_in                     100.000000
frame_id_vfr_hud                100.000000
battery_capacity                100.000000
battery_charge                  100.000000
frame_id_wind_estimation        100.000000
frame_id_target_global          100.000000
battery_serial_number           100.000000
battery_location                100.000000
rudder_failure_status            95.318380
elevator_failure_status          94.876186
aileron_failure_status           85.881729
gps_raw_angular_velocity_z       45.175750
frame_id_fix                     45.175750
gps_raw_linear_velocity_x        45.175750
gps_raw_linear_velocity_y        45.175750
child_frame_id_local             45.175750
gps_

In [7]:
# Xóa các cột mà toàn bộ dữ liệu (100%) là NA
df = df.dropna(how='all', axis=1)
df = df.reset_index(drop=True)

In [8]:
columns = ['engine_failure_status', 'aileron_failure_status', 'rudder_failure_status', 'elevator_failure_status']
print(df[columns].isna().sum())
# df[columns].tail(30)
ket_qua = []

for col in columns:
    # Thêm bước kiểm tra phòng trường hợp tên cột bị viết sai chính tả hoặc không tồn tại trong DataFrame
    if col in df.columns:
        dong_dau = df[col].first_valid_index()
        dong_cuoi = df[col].last_valid_index()

        if dong_dau is not None and dong_cuoi is not None:
            # Cắt lấy đoạn dữ liệu ở giữa
            doan_giua = df[col].loc[dong_dau:dong_cuoi]
            # Kiểm tra xem có giá trị NA nào xen giữa không
            co_na_o_giua = doan_giua.isna().any()
        else:
            co_na_o_giua = "Cột trống hoàn toàn"

        ket_qua.append({
            'Tên cột': col,
            'Dòng đầu tiên có data': dong_dau,
            'Dòng cuối cùng có data': dong_cuoi,
            'Có NA xen giữa không?': co_na_o_giua
        })
    else:
        print(f"Lưu ý: Không tìm thấy cột '{col}' trong DataFrame của bạn.")

# Chuyển kết quả thành DataFrame để xem nhanh và trực quan
df_kiem_tra = pd.DataFrame(ket_qua)
print(df_kiem_tra)

engine_failure_status      269851
aileron_failure_status     594305
rudder_failure_status      659607
elevator_failure_status    656547
dtype: int64
                   Tên cột  Dòng đầu tiên có data  Dòng cuối cùng có data  \
0    engine_failure_status                      0                  692003   
1   aileron_failure_status                 397748                  513451   
2    rudder_failure_status                 357124                  389520   
3  elevator_failure_status                 293358                  345708   

   Có NA xen giữa không?  
0                   True  
1                   True  
2                  False  
3                   True  


In [10]:

df.iloc[0:10][['flight_sequence_id', 'sequence_fault_type', 'recording_timestamp', 'engine_failure_status']]

,flight_sequence_id,sequence_fault_type,recording_timestamp,engine_failure_status
0,2018-07-18-15-53-31,engine_failure,1531943810820000000,1.0
1,2018-07-18-15-53-31,engine_failure,1531943810840000000,1.0
2,2018-07-18-15-53-31,engine_failure,1531943810861000000,1.0
3,2018-07-18-15-53-31,engine_failure,1531943810865000000,1.0
4,2018-07-18-15-53-31,engine_failure,1531943810869000000,1.0
5,2018-07-18-15-53-31,engine_failure,1531943810870000000,1.0
6,2018-07-18-15-53-31,engine_failure,1531943810879000000,1.0
7,2018-07-18-15-53-31,engine_failure,1531943810880000000,1.0
8,2018-07-18-15-53-31,engine_failure,1531943810883000000,1.0
9,2018-07-18-15-53-31,engine_failure,1531943810885000000,1.0


In [15]:
nan_percentage = (df.isnull().sum() / len(df)) * 100

# Chỉ hiển thị các cột có chứa giá trị thiếu (sắp xếp giảm dần)
missing_info = nan_percentage[nan_percentage > 0].sort_values(ascending=False)
print("Các cột bị thiếu dữ liệu và tỷ lệ % khuyết thiếu:")
print(missing_info.head(30)) # Xem nhanh 30 cột thiếu nhiều nhất

Các cột bị thiếu dữ liệu và tỷ lệ % khuyết thiếu:
target_altitude                   45.17575
target_latitude                   45.17575
sensor_timestamp_target_global    45.17575
sequence_number_target_global     45.17575
gps_raw_linear_velocity_y         45.17575
gps_raw_linear_velocity_x         45.17575
sensor_timestamp_gps_vel          45.17575
sequence_number_gps_vel           45.17575
sensor_timestamp_fix              45.17575
sequence_number_fix               45.17575
compass_heading                   45.17575
relative_altitude                 45.17575
gps_local_linear_velocity_z       45.17575
gps_local_linear_velocity_y       45.17575
gps_local_linear_velocity_x       45.17575
gps_local_orientation_qw          45.17575
gps_local_orientation_qz          45.17575
gps_local_orientation_qy          45.17575
gps_local_orientation_qx          45.17575
gps_local_position_z              45.17575
gps_local_position_y              45.17575
gps_local_position_x              45.17575
gps_

# Quy trình Tiền xử lý Dữ liệu Flight Log UAV

Quy trình này thực hiện làm sạch dữ liệu chuỗi thời gian của UAV thông qua 3 bước chính:
1. **Loại bỏ trùng lặp (Duplicate Removal):** Khử các dòng bị ghi lặp do lỗi đồng bộ hoặc xung nhịp của bộ ghi log (logger).
2. **Loại bỏ các cột không biến đổi (Zero Variance Filter):** Loại bỏ các cột chứa hằng số hoặc dữ liệu rác không mang lại thông tin cho mô hình.
3. **Xử lý giá trị khuyết thiếu (NaN/Null Handling):**
   * Tính tỷ lệ khuyết thiếu của từng cột và loại bỏ các cột có tỷ lệ NaN vượt quá ngưỡng quy định (ví dụ: > 50%).
   * Với các cột còn lại, áp dụng phương pháp **Nội suy tuyến tính (Linear Interpolation)** kết hợp **Điền tiến/lùi (Forward/Backward Fill)** ở các biên dữ liệu nhằm bảo toàn tính liên tục của chuỗi thời gian.

In [12]:
def preprocess_uav_flight_data(df: pd.DataFrame, nan_threshold: float = 0.5) -> pd.DataFrame:
    """Tiền xử lý dữ liệu log UAV bằng cách loại bỏ trùng lặp,

    lọc bỏ các cột không biến đổi (phương sai bằng 0), và nội suy các giá trị thiếu.

    Tham số:
    ----------
    df : pd.DataFrame
        DataFrame chứa dữ liệu log UAV đầu vào.
    nan_threshold : float, mặc định 0.5
        Ngưỡng tỷ lệ khuyết thiếu tối đa cho phép của một cột (giá trị từ 0.0 đến 1.0).
        Các cột có tỷ lệ NaN lớn hơn ngưỡng này sẽ bị loại bỏ.

    Trả về:
    -------
    pd.DataFrame
        DataFrame sau khi đã được làm sạch và xử lý hoàn thiện.
    """
    # Tạo bản sao dữ liệu để tránh ảnh hưởng đến DataFrame gốc
    df_cleaned = df.copy()
    initial_shape = df_cleaned.shape

    print("=== BẮT ĐẦU QUY TRÌNH TIỀN XỬ LÝ DỮ LIỆU UAV ===")
    print(f"Kích thước ban đầu: {initial_shape[0]} dòng, {initial_shape[1]} cột\n")

    # -------------------------------------------------------------------------
    # BƯỚC 1: KIỂM TRA VÀ LOẠI BỎ CÁC DÒNG TRÙNG LẶP (DUPLICATES)
    # -------------------------------------------------------------------------
    num_duplicates = df_cleaned.duplicated().sum()
    if num_duplicates > 0:
        df_cleaned = df_cleaned.drop_duplicates()
        print(f"[Bước 1] Đã loại bỏ {num_duplicates} dòng trùng lặp do lỗi xung nhịp bộ ghi log.")
    else:
        print("[Bước 1] Không phát hiện dòng trùng lặp nào trong dữ liệu.")

    # -------------------------------------------------------------------------
    # BƯỚC 2: LOẠI BỎ CÁC CỘT KHÔNG BIẾN ĐỔI (ZERO VARIANCE)
    # -------------------------------------------------------------------------
    # Xác định các cột chỉ chứa duy nhất một giá trị (nunique == 1)
    # Cách tiếp cận này hoạt động ổn định trên cả kiểu dữ liệu số và phi số
    constant_cols = [col for col in df_cleaned.columns if df_cleaned[col].nunique() <= 1]

    if len(constant_cols) > 0:
        df_cleaned = df_cleaned.drop(columns=constant_cols)
        print(f"[Bước 2] Đã loại bỏ {len(constant_cols)} cột có phương sai bằng 0 (hằng số).")
    else:
        print("[Bước 2] Không có cột nào có phương sai bằng 0.")

    # -------------------------------------------------------------------------
    # BƯỚC 3: XỬ LÝ GIÁ TRỊ THIẾU (NAN/NULL)
    # -------------------------------------------------------------------------
    # 3.1. Tính toán tỷ lệ khuyết thiếu của mỗi cột
    missing_ratios = df_cleaned.isnull().mean()

    # 3.2. Lọc bỏ các cột có tỷ lệ NaN cao hơn ngưỡng quy định (ví dụ: > 50%)
    high_nan_cols = missing_ratios[missing_ratios > nan_threshold].index.tolist()

    if len(high_nan_cols) > 0:
        df_cleaned = df_cleaned.drop(columns=high_nan_cols)
        print(f"[Bước 3.1] Đã loại bỏ {len(high_nan_cols)} cột có tỷ lệ NaN > {nan_threshold*100}%.")
    else:
        print(f"[Bước 3.1] Không có cột nào có tỷ lệ NaN vượt ngưỡng {nan_threshold*100}%.")

    # 3.3. Áp dụng phương pháp nội suy cho các giá trị thiếu còn lại
    # Tìm các cột còn chứa giá trị khuyết thiếu sau bước lọc
    remaining_nan_cols = df_cleaned.columns[
        df_cleaned.isnull().any()
    ].tolist()

    # if len(remaining_nan_cols) > 0:
    #     # Sử dụng nội suy tuyến tính (Linear Interpolation) phù hợp cho chuỗi thời gian liên tục
    #     df_cleaned[remaining_nan_cols] = df_cleaned[
    #         remaining_nan_cols
    #     ].interpolate(method="linear", limit_direction="both")

    #     # Điền các giá trị thiếu còn sót lại ở vùng biên (đầu hoặc cuối chuỗi)
    #     # bằng cách kết hợp Forward Fill và Backward Fill
    #     df_cleaned[remaining_nan_cols] = (
    #         df_cleaned[remaining_nan_cols].ffill().bfill()
    #     )
    #     print(
    #         f"[Bước 3.2] Đã áp dụng nội suy tuyến tính và xử lý triệt để NaN cho {len(remaining_nan_cols)} cột."
    #     )
    # else:
    #     print(
    #         "[Bước 3.2] Không còn giá trị thiếu (NaN) nào cần xử lý trong tập dữ liệu."
    #     )

    # -------------------------------------------------------------------------
    # KẾT QUẢ CUỐI CÙNG
    # -------------------------------------------------------------------------
    final_shape = df_cleaned.shape
    print("\n=== QUY TRÌNH HOÀN TẤT ===")
    print(
        f"Kích thước sau khi xử lý: {final_shape[0]} dòng, {final_shape[1]} cột"
    )
    print(
        f"Đã giảm: {initial_shape[0] - final_shape[0]} dòng và {initial_shape[1] - final_shape[1]} cột."
    )

    return df_cleaned

df = preprocess_uav_flight_data(df)

=== BẮT ĐẦU QUY TRÌNH TIỀN XỬ LÝ DỮ LIỆU UAV ===
Kích thước ban đầu: 692004 dòng, 433 cột

[Bước 1] Không phát hiện dòng trùng lặp nào trong dữ liệu.
[Bước 2] Đã loại bỏ 204 cột có phương sai bằng 0 (hằng số).
[Bước 3.1] Đã loại bỏ 2 cột có tỷ lệ NaN > 50.0%.

=== QUY TRÌNH HOÀN TẤT ===
Kích thước sau khi xử lý: 692004 dòng, 227 cột
Đã giảm: 0 dòng và 206 cột.


In [13]:
nan_percentage = (df.isnull().sum() / len(df)) * 100

# Chỉ hiển thị các cột có chứa giá trị thiếu (sắp xếp giảm dần)
missing_info = nan_percentage[nan_percentage > 0].sort_values(ascending=False)
print("Các cột bị thiếu dữ liệu và tỷ lệ % khuyết thiếu:")
print(missing_info.head(500)) # Xem nhanh 30 cột thiếu nhiều nhất

Các cột bị thiếu dữ liệu và tỷ lệ % khuyết thiếu:
target_altitude                   45.175750
target_latitude                   45.175750
sensor_timestamp_target_global    45.175750
sequence_number_target_global     45.175750
gps_raw_linear_velocity_y         45.175750
                                    ...    
diag_3_val_value_7                 1.336264
diag_3_val_value_1                 1.336264
diag_message_3                     1.336264
diag_3_val_key_18                  1.336264
diag_3_val_value_18                1.336264
Length: 223, dtype: float64
